<a href="https://colab.research.google.com/github/Ashik-13/PINNs/blob/main/Solar_PV_Thermal_Surrogate_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Solar PV Thermal Surrogate Modeling

## Machine Learning Surrogate for Fast Prediction of PV Module Temperature

---

## Overview

Solar photovoltaic (PV) modules convert incident solar radiation into
electrical energy. However, only a fraction of the incident solar energy
is converted into electricity. The remaining energy contributes to thermal
heating of the PV module.

The operating temperature of a PV module is an important parameter because
PV electrical efficiency generally decreases as module temperature increases.

In this project, a simplified physics-based thermal model is first developed
to predict the temperature of a PV module from environmental conditions.

The physics model is then used to generate a synthetic dataset.

A Machine Learning model is subsequently trained on this dataset to construct
a computationally efficient **surrogate model**.

The fundamental mapping is

$$
\boxed{
(G,T_a,v)
\rightarrow
T_{PV}
}
$$

where:

- $G$ = solar irradiance $[\mathrm{W/m^2}]$
- $T_a$ = ambient temperature $[^\circ C]$
- $v$ = wind speed $[\mathrm{m/s}]$
- $T_{PV}$ = PV module temperature $[^\circ C]$


## Objectives

The main objectives of this notebook are to:

- formulate a simplified thermal model of a solar PV module,
- define the governing energy-balance equation,
- calculate PV module temperature from environmental conditions,
- generate a physics-based synthetic dataset,
- train a Machine Learning surrogate model,
- evaluate surrogate prediction accuracy,
- compare surrogate predictions with the physics model,
- investigate prediction errors,
- test the surrogate under new environmental conditions,
- compare the computational cost of the physics model and surrogate.

## Surrogate Modeling Workflow

The overall workflow is

$$
G,\;T_a,\;v
$$

$$
\downarrow
$$

$$
\boxed{\text{Physics-Based Thermal Model}}
$$

$$
\downarrow
$$

$$
T_{PV}
$$

$$
\downarrow
$$

$$
\boxed{\text{Synthetic Dataset}}
$$

$$
\downarrow
$$

$$
\boxed{\text{Machine Learning Surrogate}}
$$

$$
\downarrow
$$

$$
\hat{T}_{PV}
$$

$$
\downarrow
$$

$$
\boxed{\text{Validation}}
$$


## Important Idea

The Machine Learning model does not initially learn directly from experimental
measurements.

Instead, the physics-based model is used as a computational data generator.

Therefore,

$$
\boxed{
\text{Physics Model}
\rightarrow
\text{Training Data}
\rightarrow
\text{Surrogate Model}
}
$$

Once trained, the surrogate provides a fast approximation of the physics model.

This creates the foundation for later applications such as:

- fast simulation,
- parametric studies,
- optimization,
- uncertainty analysis,
- real-time prediction,
- digital twins,
- and physics-informed machine learning.

# Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from scipy.optimize import brentq

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("NumPy version      :", np.__version__)
print("Pandas version     :", pd.__version__)
print("TensorFlow version :", tf.__version__)

NumPy version      : 2.1.3
Pandas version     : 2.2.3
TensorFlow version : 2.20.0


# Physical Problem

## Thermal Behavior of a Solar PV Module

A solar photovoltaic module receives solar radiation from the sun.

The incident solar irradiance is represented by

$$
G\;[\mathrm{W/m^2}]
$$

A portion of this incident energy is converted into electrical power.

The remaining absorbed energy contributes to heating of the PV module.

The module subsequently loses heat through mechanisms such as:

- convection to the surrounding air,
- thermal radiation to the surroundings.

Therefore, the PV module temperature is determined by the balance between
absorbed thermal energy and heat losses.

## Environmental Inputs

For the present simplified model, three environmental variables are used:

$$
\boxed{
G,\;T_a,\;v
}
$$

where:

- $G$ = solar irradiance,
- $T_a$ = ambient temperature,
- $v$ = wind speed.

The quantity to be predicted is

$$
\boxed{
T_{PV}
}
$$

Therefore, the thermal problem can be written as

$$
\boxed{
T_{PV}=f(G,T_a,v)
}
$$


## Why PV Temperature Matters

The electrical efficiency of a photovoltaic module is temperature dependent.

A higher operating temperature generally results in lower PV electrical
conversion efficiency.

Therefore, PV temperature prediction is important for:

- PV performance analysis,
- energy-yield estimation,
- thermal management,
- system optimization,
- performance monitoring,
- digital-twin development.


## Modeling Objective

The main objective is to construct a computationally efficient approximation
of the mapping

$$
\boxed{
(G,T_a,v)
\rightarrow
T_{PV}
}
$$

The physics model will first provide the reference solution.

A Machine Learning model will then learn this relationship and act as a
surrogate.

# Governing Energy-Balance Equation

## Thermal Energy Balance

The PV module is modeled using a simplified steady-state energy balance.

The absorbed solar energy that is not converted into electricity contributes
to thermal heating.

Therefore,

$$
\boxed{
Q_{\mathrm{thermal}}
=
Q_{\mathrm{conv}}
+
Q_{\mathrm{rad}}
}
$$

where:

- $Q_{\mathrm{thermal}}$ = thermal energy generated in the module,
- $Q_{\mathrm{conv}}$ = convective heat loss,
- $Q_{\mathrm{rad}}$ = radiative heat loss.


## Absorbed Thermal Energy

The incident solar power is

$$
Q_{\mathrm{solar}}=\alpha GA
$$

where:

- $\alpha$ = solar absorptivity,
- $G$ = solar irradiance,
- $A$ = module area.

A fraction of the absorbed energy is converted into electricity.

Therefore,

$$
\boxed{
Q_{\mathrm{thermal}}
=
\alpha GA(1-\eta_{PV})
}
$$

where $\eta_{PV}$ is the electrical efficiency of the PV module.

## Convective Heat Loss

Convective heat transfer is modeled as

$$
\boxed{
Q_{\mathrm{conv}}
=
hA(T_{PV}-T_a)
}
$$

where $h$ is the convective heat-transfer coefficient.

The wind speed is assumed to influence convection through

$$
\boxed{
h=h_0+C_vv
}
$$

where:

- $h_0$ = base convection coefficient,
- $C_v$ = wind-dependent coefficient,
- $v$ = wind speed.

## Radiative Heat Loss

Thermal radiation is modeled using the Stefan-Boltzmann relation:

$$
\boxed{
Q_{\mathrm{rad}}
=
\epsilon\sigma A
\left[
(T_{PV}+273.15)^4
-
(T_a+273.15)^4
\right]
}
$$

where:

- $\epsilon$ = surface emissivity,
- $\sigma$ = Stefan-Boltzmann constant,
- temperatures are converted from Celsius to Kelvin.

## Temperature-Dependent PV Efficiency

The electrical efficiency is approximated by

$$
\boxed{
\eta_{PV}
=
\eta_{ref}
\left[
1-\beta(T_{PV}-T_{ref})
\right]
}
$$

where:

- $\eta_{ref}$ = reference electrical efficiency,
- $\beta$ = temperature coefficient,
- $T_{ref}$ = reference temperature.

## Complete Energy Balance

Combining the terms gives

$$
\boxed{
\alpha GA(1-\eta_{PV})
=
hA(T_{PV}-T_a)
+
\epsilon\sigma A
\left[
(T_{PV}+273.15)^4
-
(T_a+273.15)^4
\right]
}
$$

This nonlinear equation will be solved numerically for

$$
\boxed{
T_{PV}
}
$$

for every combination of environmental conditions.

## Physics Model

Thus,

$$
\boxed{
(G,T_a,v)
\rightarrow
T_{PV}
}
$$

is obtained by solving the thermal energy-balance equation.

# Define Physical Parameters

In [ ]:
# PV module properties
A = 1.6                  # Module area [m²]
alpha = 0.90             # Solar absorptivity [-]
epsilon = 0.90           # Thermal emissivity [-]

# PV electrical properties
eta_ref = 0.20           # Reference electrical efficiency [-]
beta = 0.004             # Temperature coefficient [1/°C]
T_ref = 25.0             # Reference temperature [°C]

# Convection parameters
h0 = 5.0                 # Base convection coefficient [W/m²-K]
C_v = 4.0                # Wind-dependent coefficient [W/m²-K per m/s]

# Stefan-Boltzmann constant
sigma = 5.670374419e-8   # [W/m²-K⁴]

# Environmental ranges
G_min = 200.0            # [W/m²]
G_max = 1000.0           # [W/m²]

Ta_min = 15.0            # [°C]
Ta_max = 40.0            # [°C]

v_min = 0.5              # [m/s]
v_max = 8.0              # [m/s]

print("PV Physical Parameters")
print("-----------------------------")
print(f"Area                   : {A} m²")
print(f"Absorptivity           : {alpha}")
print(f"Emissivity             : {epsilon}")
print(f"Reference efficiency   : {eta_ref}")
print(f"Temperature coefficient: {beta} 1/°C")
print(f"Reference temperature  : {T_ref} °C")
print(f"Base convection h0     : {h0} W/m²-K")
print(f"Wind coefficient Cv    : {C_v} W/m²-K per m/s")

PV Physical Parameters
-----------------------------
Area                  : 1.6 m²
Absorptivity           : 0.9
Emissivity             : 0.9
Reference efficiency   : 0.2
Temperature coefficient: 0.004 1/°C
Reference temperature  : 25.0 °C
Base convection h0     : 5.0 W/m²-K
Wind coefficient Cv    : 4.0 W/m²-K per m/s


# Model Assumptions

To keep the first surrogate-modeling problem physically meaningful while
remaining computationally manageable, the following assumptions are used.

### Assumption 1 — Steady-State Thermal Condition

The PV module is assumed to reach a steady thermal condition.

Therefore,

$$
\frac{dT_{PV}}{dt}=0
$$

and the problem becomes an algebraic energy-balance problem.

### Assumption 2 — Uniform Module Temperature

The PV module is represented by a single representative temperature:

$$
T_{PV}
$$

Spatial temperature variations across the module are neglected.

### Assumption 3 — Wind-Dependent Convection

The convective heat-transfer coefficient is approximated by

$$
h=h_0+C_vv.
$$

This provides a simple relationship between wind speed and convective cooling.


### Assumption 4 — Temperature-Dependent Electrical Efficiency

PV efficiency is approximated using

$$
\eta_{PV}
=
\eta_{ref}
[1-\beta(T_{PV}-T_{ref})].
$$

### Assumption 5 — Radiation to Ambient Environment

Radiative heat exchange is approximated using the ambient temperature as
the effective surrounding temperature.

## Scope of the First Model

This is a simplified research/educational thermal model.

It is not intended to reproduce every physical process occurring in a real
PV module.

The purpose is to establish a clean relationship between

$$
G,\;T_a,\;v
$$

and

$$
T_{PV}
$$

that can later be approximated using Machine Learning.

# Physics-Based PV Temperature Model

The physics model must determine the PV module temperature for a given set
of environmental conditions.

For a specified

$$
G,\;T_a,\;v,
$$

the unknown quantity is

$$
T_{PV}.
$$

The governing equation is

$$
\alpha GA(1-\eta_{PV})
=
hA(T_{PV}-T_a)
+
\epsilon\sigma A
\left[
(T_{PV}+273.15)^4
-
(T_a+273.15)^4
\right].
$$

Since

$$
\eta_{PV}
=
\eta_{ref}
[1-\beta(T_{PV}-T_{ref})],
$$

and

$$
h=h_0+C_vv,
$$

the equation becomes nonlinear in $T_{PV}$.

Therefore, a numerical root-finding method will be used.

## Residual Form

Define the thermal residual as

$$
R(T_{PV})
=
Q_{\mathrm{thermal}}
-
Q_{\mathrm{conv}}
-
Q_{\mathrm{rad}}.
$$

The physical solution satisfies

$$
\boxed{
R(T_{PV})=0.
}
$$

The numerical solver therefore searches for the temperature satisfying

$$
\boxed{
R(T_{PV})=0.
}
$$

# Implement Physics Model

In [ ]:
def pv_temperature_residual(T_pv, G, T_amb, wind_speed):

    # Wind-dependent convection coefficient
    h = h0 + C_v * wind_speed

    # Temperature-dependent electrical efficiency
    eta_pv = eta_ref * (1.0 - beta * (T_pv - T_ref))

    # Thermal energy generated
    Q_thermal = alpha * G * A * (1.0 - eta_pv)

    # Convective heat loss
    Q_conv = h * A * (T_pv - T_amb)

    # Radiative heat loss
    T_pv_K = T_pv + 273.15
    T_amb_K = T_amb + 273.15

    Q_rad = (epsilon * sigma * A * (T_pv_K**4 - T_amb_K**4))

    # Energy-balance residual
    residual = Q_thermal - Q_conv - Q_rad

    return residual

def calculate_pv_temperature(G, T_amb, wind_speed):

    # Temperature search interval
    T_lower = T_amb
    T_upper = T_amb + 150.0

    T_pv = brentq(pv_temperature_residual, T_lower, T_upper, args=(G, T_amb, wind_speed))

    return T_pv

# Test the Physics Model

In [ ]:
G_test = 800.0
Ta_test = 30.0
v_test = 2.0

T_pv_test = calculate_pv_temperature(G_test, Ta_test, v_test)

print("Physics Model Test")
print("-----------------------------")
print(f"Irradiance       : {G_test:.1f} W/m²")
print(f"Ambient Temp.    : {Ta_test:.1f} °C")
print(f"Wind Speed       : {v_test:.1f} m/s")
print(f"PV Temperature   : {T_pv_test:.3f} °C")

Physics Model Test
-----------------------------
Irradiance       : 800.0 W/m²
Ambient Temp.    : 30.0 °C
Wind Speed       : 2.0 m/s
PV Temperature   : 60.426 °C


# Physics Model Verification

Before generating a large dataset, the physical model should be examined
under several environmental conditions.

Three physical trends are particularly important.

## Effect of Solar Irradiance

Increasing irradiance should generally increase the PV module temperature.

Therefore,

$$
G\uparrow
\quad\Rightarrow\quad
T_{PV}\uparrow
$$

## Effect of Ambient Temperature

Increasing ambient temperature should increase the PV module temperature.

Therefore,

$$
T_a\uparrow
\quad\Rightarrow\quad
T_{PV}\uparrow
$$

## Effect of Wind Speed

Increasing wind speed increases convective heat transfer.

Therefore, stronger wind should generally enhance cooling:

$$
v\uparrow
\quad\Rightarrow\quad
h\uparrow
$$

and consequently,

$$
T_{PV}\downarrow
$$

for otherwise identical conditions.

These trends provide an important sanity check before using the model
to generate the surrogate dataset.

# Interactive Physics Trend Verification

In [ ]:
# Fixed conditions
Ta_fixed = 30.0
v_fixed = 2.0

irradiance_values = np.linspace(G_min, G_max,100)

temperature_vs_G = [calculate_pv_temperature(G, Ta_fixed, v_fixed) for G in irradiance_values]

fig = go.Figure()

fig.add_trace(go.Scatter(x=irradiance_values, y=temperature_vs_G, mode="lines", name="PV Temperature"))

fig.update_layout(
    title="PV Temperature vs Solar Irradiance",
    xaxis_title="Solar Irradiance (W/m²)",
    yaxis_title="PV Temperature (°C)",
    template="plotly_white"
)

fig.show()

# Wind-Speed Verification

PV Temperature vs Wind Speed

In [ ]:
G_fixed = 800.0
Ta_fixed = 30.0

wind_values = np.linspace(v_min, v_max, 100)

temperature_vs_wind = [calculate_pv_temperature(G_fixed, Ta_fixed, v) for v in wind_values]

fig = go.Figure()

fig.add_trace(go.Scatter(x=wind_values, y=temperature_vs_wind, mode="lines", name="PV Temperature"))

fig.update_layout(
    title="PV Temperature vs Wind Speed",
    xaxis_title="Wind Speed (m/s)",
    yaxis_title="PV Temperature (°C)",
    template="plotly_white"
)

fig.show()

# Synthetic Dataset Generation

The physics model can now be used as a data generator.

We randomly sample environmental conditions from the defined ranges:

$$
200\leq G\leq1000\;\mathrm{W/m^2}
$$

$$
15\leq T_a\leq40^\circ C
$$

$$
0.5\leq v\leq8\;\mathrm{m/s}
$$

For every sampled combination,

$$
(G_i,T_{a,i},v_i),
$$

the physics model calculates

$$
T_{PV,i}.
$$

Therefore, each dataset sample becomes

$$
\boxed{
[G_i,\;T_{a,i},\;v_i]
\rightarrow
T_{PV,i}
}
$$

The resulting dataset is generated entirely from the physics model.

## Dataset Structure

The final dataset will contain four columns:
<div align="center">

| Input / Output | Variable | Unit |
|---|---|---|
| Input | Irradiance | W/m² |
| Input | Ambient Temperature | °C |
| Input | Wind Speed | m/s |
| Output | PV Temperature | °C |

</div>

This dataset will subsequently be used to train the surrogate model.

# Generate Synthetic Dataset

In [ ]:
N_samples = 10000

rng = np.random.default_rng(42)

G_data = rng.uniform(G_min, G_max, N_samples)

Ta_data = rng.uniform(Ta_min, Ta_max, N_samples)

v_data = rng.uniform(v_min, v_max, N_samples)

# Calculate physics-model temperature
T_pv_data = np.array([
    calculate_pv_temperature(G,Ta,v)
    for G, Ta, v in zip(G_data, Ta_data, v_data)
])

# Construct DataFrame
data = pd.DataFrame({
    "Irradiance_W_m2": G_data,
    "Ambient_Temperature_C": Ta_data,
    "Wind_Speed_m_s": v_data,
    "PV_Temperature_C": T_pv_data
})

print("Dataset shape:", data.shape)

data.head()

Dataset shape: (10000, 4)


,Irradiance_W_m2,Ambient_Temperature_C,Wind_Speed_m_s,PV_Temperature_C
0,819.164839,33.018210,6.055796,50.005752
1,551.102752,32.780944,7.968548,42.168498
2,886.878336,20.067376,0.807330,65.278217
3,757.894423,15.916385,2.336896,43.661660
4,275.341878,22.594988,4.983056,29.145290


# Dataset Exploration

Before training a Machine Learning model, the generated dataset should be
examined.

We will investigate:

- dataset dimensions,
- descriptive statistics,
- input distributions,
- output distribution,
- relationships between environmental variables and PV temperature.

The purpose is to ensure that the dataset is physically reasonable and
contains sufficient variation for surrogate-model training.

# Dataset Statistics

In [ ]:
print("Dataset Information")
print("============================")

print("\nShape:")
print(data.shape)

print("\nDescriptive Statistics:")
display(data.describe())

print("\nMissing Values:")
print(data.isnull().sum())

Dataset Information

Shape:
(10000, 4)

Descriptive Statistics:


,Irradiance_W_m2,Ambient_Temperature_C,Wind_Speed_m_s,PV_Temperature_C
count,10000.000000,10000.000000,10000.000000,10000.000000
mean,597.697278,27.573392,4.268613,44.899355
std,230.576736,7.223571,2.163676,11.659840
min,200.251536,15.000329,0.500354,19.520181
25%,396.830392,21.249411,2.398560,36.426944
50%,598.641193,27.691339,4.265853,44.140887
75%,796.542159,33.798521,6.179462,51.994393
max,999.981616,39.998681,7.999869,87.485719



Missing Values:
Irradiance_W_m2          0
Ambient_Temperature_C    0
Wind_Speed_m_s           0
PV_Temperature_C         0
dtype: int64


# Interactive Dataset Visualization

In [ ]:
fig = px.scatter_3d(
    data,
    x="Irradiance_W_m2",
    y="Ambient_Temperature_C",
    z="PV_Temperature_C",
    color="Wind_Speed_m_s",
    title="PV Temperature Dataset",
    labels={
        "Irradiance_W_m2": "Irradiance (W/m²)",
        "Ambient_Temperature_C": "Ambient Temperature (°C)",
        "Wind_Speed_m_s": "Wind Speed (m/s)",
        "PV_Temperature_C": "PV Temperature (°C)"
    }
)

fig.show()

# Input–Output Formulation

The surrogate model will learn the mapping

$$
\boxed{
X\rightarrow y
}
$$

where the input matrix is

$$
\boxed{
X=
\begin{bmatrix}
G & T_a & v
\end{bmatrix}
}
$$

and the target is

$$
\boxed{
y=T_{PV}
}
$$

Therefore,

### Inputs

$$
X=
[G,\;T_a,\;v]
$$

### Output

$$
y=T_{PV}
$$

The Machine Learning model will approximate

$$
\boxed{
\hat{T}_{PV}
=
f_{\theta}(G,T_a,v)
}
$$

where $\theta$ represents the trainable parameters of the surrogate model.

## Important Distinction

The physics model generated the training data.

The Machine Learning model will now learn an approximation of the mapping
without explicitly solving the nonlinear energy-balance equation for every
new prediction.

# Prepare Inputs and Target

In [ ]:
# Input-Output Definition

X = data[["Irradiance_W_m2", "Ambient_Temperature_C", "Wind_Speed_m_s"]].values

y = data["PV_Temperature_C"].values.reshape(-1, 1)

print("Input shape :", X.shape)
print("Target shape:", y.shape)

Input shape : (10000, 3)
Target shape: (10000, 1)


# Train/Test Split

The available dataset is divided into two subsets:

### Training Set

Used to train the surrogate model.

### Test Set

Used only after training to evaluate how well the model generalizes to
unseen environmental conditions.

A typical split is

$$
80\%
$$

for training and

$$
20\%
$$

for testing.

The test data are not used during optimization.

# Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.20, random_state=42)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

Training samples: 8000
Testing samples : 2000


# Feature Scaling

The three input variables have different numerical scales.

For example:

$$
G\sim10^2-10^3
$$

while

$$
v\sim10^0.
$$

Neural networks generally train more effectively when input features are
normalized to comparable scales.

Standardization is performed using

$$
\boxed{
x_{scaled}
=
\frac{x-\mu}{\sigma}
}
$$

where:

- $\mu$ = training-set mean,
- $\sigma$ = training-set standard deviation.

Importantly, the scaler is fitted **only on the training data**.

The same transformation is then applied to the test data.

# Feature Scaling

In [ ]:
X_scaler = StandardScaler()

X_train_scaled = X_scaler.fit_transform(X_train)

X_test_scaled = X_scaler.transform(X_test)

print("Feature scaling completed.")

Feature scaling completed.


# ANN Surrogate Formulation

The first surrogate model will be a fully connected Artificial Neural
Network (ANN).

The ANN approximates

$$
\boxed{
(G,T_a,v)
\rightarrow
\hat{T}_{PV}
}
$$

The architecture is

$$
3
\rightarrow
64
\rightarrow
64
\rightarrow
64
\rightarrow
1
$$

where:

- 3 input neurons correspond to $G$, $T_a$, and $v$,
- three hidden layers contain 64 neurons each,
- one output neuron predicts PV temperature.

The hidden layers use the `tanh` activation function.

## Why ANN?

An ANN is suitable for this problem because the physics model defines a
nonlinear mapping between environmental conditions and PV temperature.

The ANN attempts to approximate this nonlinear function:

$$
T_{PV}=f(G,T_a,v).
$$

After training, the ANN acts as a surrogate for the original physics model.

# Create ANN Surrogate

In [ ]:
# ANN Surrogate Model

def create_surrogate_model():

    model = keras.Sequential([

        keras.Input(shape=(3,)),

        layers.Dense(64, activation="tanh"),

        layers.Dense(64, activation="tanh"),

        layers.Dense(64, activation="tanh"),

        layers.Dense(1)
    ])

    return model


surrogate = create_surrogate_model()

surrogate.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,641 (33.75 KB)

 Trainable params: 8,641 (33.75 KB)

 Non-trainable params: 0 (0.00 B)

# ANN Training Strategy

The ANN is trained using supervised learning.

For each training sample,

$$
X_i=
[G_i,T_{a,i},v_i]
$$

and the physics model provides the corresponding target

$$
y_i=T_{PV,i}.
$$

The ANN produces

$$
\hat{y}_i.
$$

The training objective is to minimize the difference between the physics
model output and the surrogate prediction.

## Mean Squared Error Loss

The primary training loss is

$$
\boxed{
\mathcal{L}
=
\frac{1}{N}
\sum_{i=1}^{N}
(y_i-\hat{y}_i)^2
}
$$

which is the Mean Squared Error (MSE).

## Optimization

The Adam optimizer will be used to update the trainable neural-network
parameters.

The model will be trained using a validation subset so that training and
validation behavior can be monitored simultaneously.

# Compile and Train ANN

In [ ]:
# Compile ANN

surrogate.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse", metrics=["mae"])

# Train ANN

history = surrogate.fit(X_train_scaled, y_train, validation_split=0.20, epochs=500, batch_size=64, verbose=1)

Epoch 1/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1585.7136 - mae: 38.0405 - val_loss: 883.7056 - val_mae: 27.3768
Epoch 2/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 706.3268 - mae: 23.7175 - val_loss: 533.1732 - val_mae: 20.0309
Epoch 3/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 445.7193 - mae: 17.7305 - val_loss: 345.0716 - val_mae: 15.1709
Epoch 4/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 297.8581 - mae: 13.7402 - val_loss: 234.5693 - val_mae: 11.7357
Epoch 5/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 207.7555 - mae: 10.8735 - val_loss: 168.5979 - val_mae: 9.5563
Epoch 6/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 151.2594 - mae: 8.8059 - val_loss: 120.8828 - val_mae: 7.3662
Epoch 7/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 109.4262 - mae: 6.6435 - val_loss: 88.2695 - val_mae: 5.6612
Epoch 8/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 82.2967 - mae: 5.3494 - val_loss: 67.5149 - val_mae: 4.6712
Epoch 9/5

# Training-Loss Analysis

During ANN training, two quantities are monitored:

- training loss,
- validation loss.

The training loss measures how well the network fits the training data.

The validation loss measures performance on data that were not directly
used to update the network parameters.

A desirable training process should show decreasing loss without a large
divergence between training and validation performance.

The loss history therefore provides an important diagnostic for the
surrogate model.

In [ ]:
# Training History

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=history.history["loss"],
        mode="lines",
        name="Training Loss"
    )
)

fig.add_trace(
    go.Scatter(
        y=history.history["val_loss"],
        mode="lines",
        name="Validation Loss"
    )
)

fig.update_layout(
    title="ANN Surrogate Training History",
    xaxis_title="Epoch",
    yaxis_title="MSE Loss",
    yaxis_type="log",
    template="plotly_white"
)

fig.show()

# Surrogate Prediction

After training, the ANN can predict PV temperature for unseen environmental
conditions.

For the test dataset,

$$
X_{test}
=
[G,T_a,v]
$$

the surrogate produces

$$
\boxed{
\hat{T}_{PV}
}
$$

These predictions will be compared against the original physics-model
outputs.

The physics-model temperature acts as the reference solution for evaluating
the surrogate.

In [ ]:
# Surrogate Prediction

y_pred = surrogate.predict(X_test_scaled, verbose=0)

print("Prediction completed.")

print("\nFirst 10 predictions:")
print(y_pred[:10].flatten())

Prediction completed.

First 10 predictions:
[46.48725  52.682644 29.227745 50.296185 39.582836 46.077854 64.70944
 37.82371  35.347652 62.628498]


# Quantitative Error Analysis

The surrogate model is evaluated using several standard regression metrics.

## Mean Absolute Error

$$
\boxed{
MAE
=
\frac{1}{N}
\sum_{i=1}^{N}
|y_i-\hat{y}_i|
}
$$

## Root Mean Squared Error

$$
\boxed{
RMSE
=
\sqrt{
\frac{1}{N}
\sum_{i=1}^{N}
(y_i-\hat{y}_i)^2
}
}
$$

## Coefficient of Determination

$$
\boxed{
R^2
=
1-
\frac{
\sum_i(y_i-\hat{y}_i)^2
}{
\sum_i(y_i-\bar{y})^2
}
}
$$

## Interpretation

A good surrogate should have:

- low MAE,
- low RMSE,
- an $R^2$ value close to 1.

These metrics quantify how closely the ANN reproduces the physics-model
temperature.

In [ ]:
# Error Metrics

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

r2 = r2_score(y_test, y_pred)

print("ANN Surrogate Performance")
print("==============================")
print(f"MAE  : {mae:.6f} °C")
print(f"RMSE : {rmse:.6f} °C")
print(f"R²   : {r2:.6f}")

ANN Surrogate Performance
MAE  : 0.068250 °C
RMSE : 0.092339 °C
R²   : 0.999931


# Actual vs Predicted PV Temperature

A useful visualization for regression models is the actual-versus-predicted
plot.

The horizontal axis represents the physics-model temperature:

$$
T_{PV}^{physics}
$$

while the vertical axis represents the surrogate prediction:

$$
T_{PV}^{surrogate}.
$$

If the surrogate reproduces the physics model accurately, the points should
lie close to the ideal diagonal:

$$
\boxed{
y=x
}
$$

Therefore, the closer the points are to the diagonal line, the closer the
surrogate predictions are to the physics-model outputs.

In [ ]:
# Actual vs Predicted

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=y_test.flatten(),
        y=y_pred.flatten(),
        mode="markers",
        name="Surrogate Predictions",
        marker=dict(size=5)
    )
)

# Ideal y = x line
min_temp = min(
    y_test.min(),
    y_pred.min()
)

max_temp = max(
    y_test.max(),
    y_pred.max()
)

fig.add_trace(
    go.Scatter(
        x=[min_temp, max_temp],
        y=[min_temp, max_temp],
        mode="lines",
        name="Ideal Prediction"
    )
)

fig.update_layout(
    title="Actual vs Predicted PV Temperature",
    xaxis_title="Physics Model Temperature (°C)",
    yaxis_title="Surrogate Prediction (°C)",
    template="plotly_white"
)

fig.show()

# Residual Analysis

The prediction residual is defined as

$$
\boxed{
e_i
=
T_{PV,i}^{physics}
-
T_{PV,i}^{surrogate}
}
$$

Residual analysis helps determine whether the surrogate errors are randomly
distributed or whether systematic errors remain.

For an accurate surrogate, the residuals should generally remain small across
the prediction range.

Large systematic deviations may indicate that the ANN has not sufficiently
captured the nonlinear physics relationship.

In [ ]:
# Residual Analysis

residuals = (y_test.flatten() - y_pred.flatten())

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=y_test.flatten(),
        y=residuals,
        mode="markers",
        name="Residual"
    )
)

fig.add_hline(y=0, line_dash="dash", annotation_text="Zero Residual")

fig.update_layout(
    title="Surrogate Prediction Residuals",
    xaxis_title="Physics Model Temperature (°C)",
    yaxis_title="Residual (°C)",
    template="plotly_white"
)

fig.show()

# New-Condition Validation

A surrogate model should not only reproduce the previously generated test
dataset.

It should also be evaluated on completely new environmental conditions.

For a new condition

$$
(G_{new},T_{a,new},v_{new}),
$$

the physics model provides

$$
T_{PV}^{physics},
$$

while the ANN provides

$$
T_{PV}^{surrogate}.
$$

The two predictions can then be compared directly.

This provides a practical demonstration of the surrogate concept:

$$
\boxed{
\text{New Input}
\rightarrow
\text{Fast ANN Prediction}
}
$$

instead of repeatedly solving the nonlinear physics equation.

In [ ]:
# New-Condition Validation

new_conditions = np.array([
    [300.0, 20.0, 1.0],
    [600.0, 25.0, 3.0],
    [900.0, 35.0, 5.0],
    [1000.0, 40.0, 1.0],
    [500.0, 30.0, 7.0]
])

# Physics model predictions
physics_predictions = np.array([
    calculate_pv_temperature(G, Ta, v)
    for G, Ta, v in new_conditions
]).reshape(-1, 1)

# Scale inputs
new_conditions_scaled = X_scaler.transform(new_conditions)

# ANN predictions
surrogate_predictions = surrogate.predict(new_conditions_scaled, verbose=0)

comparison = pd.DataFrame({
    "Irradiance_W_m2": new_conditions[:, 0],
    "Ambient_Temperature_C": new_conditions[:, 1],
    "Wind_Speed_m_s": new_conditions[:, 2],
    "Physics_Temperature_C": physics_predictions.flatten(),
    "Surrogate_Temperature_C": surrogate_predictions.flatten()
})

comparison["Absolute_Error_C"] = np.abs(comparison["Physics_Temperature_C"] - comparison["Surrogate_Temperature_C"])

comparison

,Irradiance_W_m2,Ambient_Temperature_C,Wind_Speed_m_s,Physics_Temperature_C,Surrogate_Temperature_C,Absolute_Error_C
0,300.0,20.0,1.0,34.992926,35.163963,0.171037
1,600.0,25.0,3.0,44.180317,44.196972,0.016655
2,900.0,35.0,5.0,56.133955,56.127911,0.006044
3,1000.0,40.0,1.0,85.524565,85.683929,0.159365
4,500.0,30.0,7.0,39.373953,39.406200,0.032247


# Physics Model vs Surrogate

The key idea of surrogate modeling is computational replacement.

The original physics model requires solving a nonlinear energy-balance
equation.

The trained ANN instead performs a direct forward evaluation:

$$
[G,T_a,v]
\rightarrow
\hat{T}_{PV}.
$$

Therefore, after training,

$$
\boxed{
\text{Physics Model}
\approx
\text{ANN Surrogate}
}
$$

The two models should produce closely matching temperature predictions
within the training domain.

The main advantage of the surrogate is expected to be computational speed,
particularly when a very large number of predictions are required.

# Interactive Comparison

In [ ]:
# Physics Model vs Surrogate

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=np.arange(len(physics_predictions)),
        y=physics_predictions.flatten(),
        mode="lines+markers",
        name="Physics Model"
    )
)

fig.add_trace(
    go.Scatter(
        x=np.arange(len(surrogate_predictions)),
        y=surrogate_predictions.flatten(),
        mode="lines+markers",
        name="ANN Surrogate"
    )
)

fig.update_layout(
    title="Physics Model vs ANN Surrogate",
    xaxis_title="New Test Condition",
    yaxis_title="PV Temperature (°C)",
    template="plotly_white"
)

fig.show()

# Computational Speed Comparison

One of the primary motivations for surrogate modeling is computational
efficiency.

The physics model must solve a nonlinear equation for every input condition.

The ANN surrogate performs only a forward neural-network evaluation.

Therefore, we compare the time required to generate a large number of
predictions.

The comparison is:

$$
\boxed{
\text{Physics Simulation Time}
\quad\text{vs}\quad
\text{Surrogate Prediction Time}
}
$$

The speed comparison demonstrates the practical motivation behind surrogate
modeling.

The surrogate does not replace the physics model during training.

Instead, the physics model is used to generate knowledge, and the trained
surrogate provides a fast approximation for repeated predictions.

In [ ]:
# Computational Speed Comparison

import time

N_benchmark = 5000

benchmark_inputs = np.column_stack([
    rng.uniform(G_min, G_max, N_benchmark),
    rng.uniform(Ta_min, Ta_max, N_benchmark),
    rng.uniform(v_min, v_max, N_benchmark)
])

# Physics model timing

start_time = time.perf_counter()

physics_benchmark = np.array([
    calculate_pv_temperature(G, Ta, v)
    for G, Ta, v in benchmark_inputs
])

physics_time = time.perf_counter() - start_time

# ANN surrogate timing

benchmark_scaled = X_scaler.transform(benchmark_inputs)

start_time = time.perf_counter()

surrogate_benchmark = surrogate.predict(benchmark_scaled, verbose=0)

surrogate_time = time.perf_counter() - start_time

# Results

speedup = physics_time / surrogate_time

print("Computational Speed Comparison")
print("====================================")
print(f"Number of predictions : {N_benchmark}")
print(f"Physics model time    : {physics_time:.6f} s")
print(f"ANN surrogate time    : {surrogate_time:.6f} s")
print(f"Approximate speedup   : {speedup:.2f} ×")

Computational Speed Comparison
Number of predictions : 5000
Physics model time    : 0.280819 s
ANN surrogate time    : 0.481665 s
Approximate speedup   : 0.58 ×


# Surrogate Model Summary

The complete surrogate modeling process can now be summarized as

$$
\boxed{
(G,T_a,v)
}
$$

$$
\downarrow
$$

$$
\boxed{
\text{Physics-Based Thermal Model}
}
$$

$$
\downarrow
$$

$$
\boxed{
T_{PV}
}
$$

$$
\downarrow
$$

$$
\boxed{
\text{Synthetic Dataset}
}
$$

$$
\downarrow
$$

$$
\boxed{
\text{ANN Training}
}
$$

$$
\downarrow
$$

$$
\boxed{
\hat{T}_{PV}
}
$$

## Surrogate Objective

The ANN learns the approximation

$$
\boxed{
\hat{T}_{PV}
=
f_{\theta}(G,T_a,v)
}
$$

to the physics relationship

$$
\boxed{
T_{PV}
=
f_{physics}(G,T_a,v).
}
$$

Therefore,

$$
\boxed{
f_{\theta}
\approx
f_{physics}
}
$$

within the region represented by the training dataset.

In [ ]:
# Final Performance Summary

summary = pd.DataFrame({
    "Metric": [
        "MAE",
        "RMSE",
        "R²",
        "Maximum Absolute Error"
    ],
    "Value": [
        mae,
        rmse,
        r2,
        np.max(np.abs(y_test - y_pred))
    ]
})

summary

,Metric,Value
0,MAE,0.068250
1,RMSE,0.092339
2,R²,0.999931
3,Maximum Absolute Error,0.370424


# Discussion and Conclusion

In this project, a Machine Learning surrogate model was developed for
predicting the thermal behavior of a solar photovoltaic module.

The thermal behavior was first represented using a simplified physics-based
energy balance.

The fundamental relationship was

$$
\boxed{
(G,T_a,v)
\rightarrow
T_{PV}
}
$$

where solar irradiance, ambient temperature, and wind speed determine the
PV module temperature.

## Physics-Based Model

The module temperature was calculated from

$$
\alpha GA(1-\eta_{PV})
=
hA(T_{PV}-T_a)
+
\epsilon\sigma A
\left[
(T_{PV}+273.15)^4
-
(T_a+273.15)^4
\right].
$$

The nonlinear equation was solved numerically for $T_{PV}$.

## Synthetic Dataset

A dataset was generated by sampling environmental conditions and evaluating
the physics model.

Thus, the dataset represented

$$
\boxed{
\text{Physics Knowledge}
\rightarrow
\text{Training Data}
}
$$

## Machine Learning Surrogate

An Artificial Neural Network was then trained to approximate the physics
relationship.

The surrogate learned

$$
\boxed{
[G,T_a,v]
\rightarrow
T_{PV}.
}
$$

## Validation

The trained surrogate was evaluated using:

- Mean Absolute Error,
- Root Mean Squared Error,
- $R^2$,
- actual-versus-predicted comparison,
- residual analysis,
- new-condition validation.

The surrogate was also compared with the original physics model in terms
of computational cost.

## Key Learning Outcomes

This project demonstrates the complete workflow of basic surrogate modeling:

1. Start with a physics-based model.

2. Define the model inputs and output.

3. Generate computational data from the physics model.

4. Train a Machine Learning model on the generated data.

5. Validate the surrogate using unseen conditions.

6. Compare the surrogate with the original physics model.

7. Evaluate whether the surrogate provides computational advantages.

## Important Research Concept

The most important idea is that the Machine Learning model is not being used
as a replacement for physics during the development stage.

Instead:

$$
\boxed{
\text{Physics}
\rightarrow
\text{Data}
\rightarrow
\text{Machine Learning}
}
$$

The resulting surrogate can then be used for rapid prediction.

# Final Concept

The complete project can be summarized as

$$
\boxed{
\text{Physics-Based Simulation}
+
\text{Machine Learning}
=
\text{Surrogate Model}
}
$$

For the present PV application:

$$
\boxed{
(G,T_a,v)
\rightarrow
T_{PV}^{physics}
\rightarrow
\text{ANN}
\rightarrow
T_{PV}^{surrogate}
}
$$

# 23. Future Research Direction 🚀

This basic ANN surrogate provides a foundation for more advanced
Physics + Machine Learning research.

The natural progression is:

## Stage 1 — ANN Surrogate

Current project:

$$
(G,T_a,v)
\rightarrow
T_{PV}
$$

using an Artificial Neural Network.

## Stage 2 — Surrogate Model Comparison

The same dataset can be used to compare:

$$
\boxed{
ANN
\quad vs \quad
XGBoost
\quad vs \quad
Gaussian\ Process
}
$$

Performance can be compared using:

- MAE,
- RMSE,
- $R^2$,
- computational cost,
- extrapolation behavior.

## Stage 3 — Physics-Informed Surrogate

The next major step is to incorporate the thermal physics directly into
the Machine Learning model.

Instead of relying only on

$$
\mathcal{L}_{data},
$$

we can construct a combined objective such as

$$
\boxed{
\mathcal{L}
=
\mathcal{L}_{data}
+
\lambda\mathcal{L}_{physics}
}
$$

where the physics loss measures violation of the energy-balance equation.

This creates a bridge between conventional surrogate modeling and
Physics-Informed Machine Learning.

## Stage 4 — Higher-Fidelity Simulation

The simplified physics model can eventually be replaced by data generated
from higher-fidelity simulation tools such as:

- CFD,
- finite-element thermal simulation,
- detailed PV system simulation.

The workflow becomes

$$
\boxed{
\text{High-Fidelity Simulation}
\rightarrow
\text{Dataset}
\rightarrow
\text{Surrogate}
}
$$

## Stage 5 — Optimization

Once a fast surrogate exists, it can be coupled with optimization methods.

For example:

$$
\boxed{
\text{Surrogate}
\rightarrow
\text{Optimization}
\rightarrow
\text{Optimal Design/Operation}
}
$$

## Stage 6 — Digital Twin

The long-term research direction is a PV thermal digital twin:

$$
\boxed{
\text{Physics}
+
\text{ML Surrogate}
+
\text{Real-Time Data}
\rightarrow
\text{Digital Twin}
}
$$

This provides a pathway from a simple Colab project toward more advanced
research in renewable-energy systems, scientific machine learning, and
digital-twin development.